# RLM vs a plain LLM call: IMF CPI (140 MB, 1.5M rows)

The same model, the same question, asked two ways. The dataset is the IMF's
Consumer Price Index dataflow, pulled from their public SDMX API: about 140 MB
of CSV, 1.5 million observation rows, 194 countries. It does not fit in any
context window.

Arm 1 pastes as much raw CSV as reasonably fits into a single LLM call.
Arm 2 gives the same model the file through `fabric_rlm`, so it can query the
data with DuckDB inside a CPython subprocess. A ground-truth cell at the end
grades both arms with a deterministic DuckDB query.

In [ ]:
%pip install -q fabric-rlm[analytics]

## Download the data

One GET against the IMF SDMX 3.0 API. No API key. About 140 MB; the IMF
server usually delivers it in under a minute. The cell skips the download if
the file is already there, and prefers Lakehouse `Files` when one is mounted.

In [ ]:
import os, urllib.request

DATA_DIR = "/lakehouse/default/Files" if os.path.isdir("/lakehouse/default/Files") else "."
DATA_PATH = os.path.join(DATA_DIR, "imf_cpi.csv")
URL = (
    "https://api.imf.org/external/sdmx/3.0/data/dataflow/IMF.STA/CPI/5.0.0/*.*.*.*.*"
    "?c%5BTIME_PERIOD%5D=ge:2017-01-01+le:2026-12-31"
)

if not os.path.exists(DATA_PATH):
    req = urllib.request.Request(URL, headers={"Accept": "application/vnd.sdmx.data+csv"})
    with urllib.request.urlopen(req, timeout=900) as r, open(DATA_PATH, "wb") as fh:
        while chunk := r.read(1 << 20):
            fh.write(chunk)
print(f"{os.path.getsize(DATA_PATH) / 1e6:.0f} MB at {DATA_PATH}")

## The task

Both arms get exactly the same question, including the same column hints, so
the only difference between them is data access.

In [ ]:
TASK = """The file is IMF CPI data in SDMX-CSV format (one observation per row).
Use monthly year-over-year CPI inflation for all items, which means rows with
INDEX_TYPE='CPI', COICOP_1999='_T', TYPE_OF_TRANSFORMATION='YOY_PCH_PA_PT',
FREQUENCY='M'. TIME_PERIOD looks like '2021-M01'; COUNTRY is an ISO3 code;
the value is in OBS_VALUE.

Consider only countries that have all 60 monthly observations for 2021
through 2025. Compute each country's average inflation over that window and
report:
- n_countries: how many countries qualify
- top5: the 5 highest averages as a list of {country, avg_yoy}, 2 decimals
- bottom5: the 5 lowest, same shape
- median_avg_yoy: the median of the qualifying country averages, 2 decimals
"""

## Arm 1: plain LLM call

The honest baseline: read as much of the file as fits comfortably in a prompt
(200,000 characters, which is under 0.15 percent of the file) and ask. The
slice covers part of one country, so watch what the model does.

In [ ]:
import time
from fabric_rlm import FabricLM

lm = FabricLM("gpt-5", reasoning_effort="minimal")

head = open(DATA_PATH, encoding="utf-8").read(200_000)
t0 = time.time()
plain_text = lm(f"{TASK}\n\nHere is as much of the file as fits:\n\n{head}")[0]
plain_seconds = time.time() - t0
plain_usage = (lm.history[-1].get("usage") or {}) if getattr(lm, "history", None) else {}
print(plain_text)

## Arm 2: the same model, through the RLM

Same model, same task. The file is bound as an input, so the model writes
DuckDB and pandas code against the full 1.5 million rows inside the
subprocess. Raw bytes never enter its context.

In [ ]:
from fabric_rlm import File, RLM

t0 = time.time()
rlm = RLM.task(
    task=TASK,
    inputs={"data_file": File(DATA_PATH)},
    outputs=["n_countries", "top5", "bottom5", "median_avg_yoy"],
    lm=FabricLM("gpt-5", reasoning_effort="minimal"),
    skills=["data_exploration"],
    max_turns=8,
)
result = rlm.run()
rlm_seconds = time.time() - t0
result.payload

## Ground truth and scoreboard

A deterministic DuckDB query grades both arms. The plain arm is graded
generously: it only has to mention the five correct top-5 country codes
anywhere in its answer.

In [ ]:
import duckdb, statistics

con = duckdb.connect()
rows = con.execute(f"""
WITH obs AS (
    SELECT COUNTRY, OBS_VALUE
    FROM read_csv_auto('{DATA_PATH}')
    WHERE INDEX_TYPE = 'CPI' AND COICOP_1999 = '_T'
      AND TYPE_OF_TRANSFORMATION = 'YOY_PCH_PA_PT' AND FREQUENCY = 'M'
      AND substr(TIME_PERIOD, 1, 4) BETWEEN '2021' AND '2025'
      AND OBS_VALUE IS NOT NULL
), per_country AS (
    SELECT COUNTRY, count(*) AS n_obs, avg(OBS_VALUE) AS avg_yoy
    FROM obs GROUP BY COUNTRY
)
SELECT COUNTRY, round(avg_yoy, 2) AS avg_yoy
FROM per_country WHERE n_obs = 60 ORDER BY avg_yoy DESC
""").fetchall()

truth = {
    "n_countries": len(rows),
    "top5": [{"country": c, "avg_yoy": v} for c, v in rows[:5]],
    "bottom5": [{"country": c, "avg_yoy": v} for c, v in sorted(rows[-5:], key=lambda r: r[1])],
    "median_avg_yoy": round(statistics.median(v for _, v in rows), 2),
}
truth

In [ ]:
def close(a, b, tol=0.05):
    try:
        return abs(float(a) - float(b)) <= tol
    except (TypeError, ValueError):
        return False

truth_top5 = [d["country"] for d in truth["top5"]]

plain_correct = all(c in plain_text for c in truth_top5)

payload = result.payload or {}
rlm_top5 = [str(d.get("country", "")) for d in (payload.get("top5") or []) if isinstance(d, dict)]
rlm_correct = (
    rlm_top5 == truth_top5
    and close(payload.get("median_avg_yoy"), truth["median_avg_yoy"])
    and payload.get("n_countries") == truth["n_countries"]
)

plain_tokens = (plain_usage.get("prompt_tokens") or 0) + (plain_usage.get("completion_tokens") or 0)
rlm_tokens = (result.total_prompt_tokens or 0) + (result.total_completion_tokens or 0)

print(f"{'arm':<18} {'correct':<9} {'tokens':>10} {'seconds':>9}")
print(f"{'plain LLM call':<18} {str(plain_correct):<9} {plain_tokens:>10,} {plain_seconds:>9.1f}")
print(f"{'RLM':<18} {str(rlm_correct):<9} {rlm_tokens:>10,} {rlm_seconds:>9.1f}")

## What just happened

The plain call saw less than 0.15 percent of the file, so the best it can do
is guess from a slice of one country plus whatever it remembers about world
inflation. The RLM arm wrote a DuckDB query, ran it over all 1.5 million rows
in the subprocess, and submitted numbers that match the deterministic ground
truth. Only the model's own code and summaries ever touched its context.

That is the whole pitch: when the answer must be computed from data that does
not fit in context, give the model an interpreter instead of a bigger prompt.
The README section "When to use an RLM (and when not to)" covers when the
trade is worth it, including the cases where it is not.